In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets
!pip install optuna optuna-integration wandb

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-z_c8d0wf/unsloth_de01f7c3d0e6433db15ceca60498ec10
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-z_c8d0wf/unsloth_de01f7c3d0e6433db15ceca60498ec10
  Resolved https://github.com/unslothai/unsloth.git to commit 9a551831ef2eff33f947cc1a7e1b259f9913950b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 155.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 22.5 MB/s eta 0:00:00

In [2]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch, wandb, os, gc
from google.colab import userdata, drive

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# 1. System Prep
drive.mount('/content/drive')
wandb.login(key=userdata.get('WANDB_API_KEY'))
os.environ["WANDB_PROJECT"] = "medical-llama3-production-16bit"
os.environ["WANDB_LOG_MODEL"] = "false" # Saves bandwidth/compute

Mounted at /content/drive


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akhilgalla (foobar41) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
# 2. Model Loading (16-bit Native)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Meta-Llama-3-8B-Instruct",
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct as a legacy tokenizer.


In [5]:
# 3. Winning LoRA Config (Trial 6 Results)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407
)

Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [6]:
# 4. Dataset Prep (34,000 Medical Flashcards)
dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")

llama3_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a highly accurate medical AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{}<|eot_id|>"""

EOS_TOKEN = tokenizer.eos_token
def format_flashcards(examples):
    texts = [llama3_prompt.format(i, o) + EOS_TOKEN for i, o in zip(examples["input"], examples["output"])]
    return { "text" : texts }

full_dataset = dataset.map(format_flashcards, batched=True, remove_columns=dataset.column_names)

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

In [7]:
# 5. Production Configuration
sft_config = SFTConfig(
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,  # Total batch = 64
    num_train_epochs=1,
    learning_rate=0.0007099375152773817, # Your Winning LR
    weight_decay=0.07328643047092469,     # Your Winning Decay
    warmup_ratio=0.1,
    fp16=False,
    bf16=True,
    optim="adamw_8bit",
    seed=3407,
    output_dir="/content/drive/MyDrive/medical-llama3/production-checkpoints",
    save_strategy="steps",
    save_steps=100,      # Frequent saves in case Colab disconnects
    report_to="wandb",
    run_name="production-run-34k",
    neftune_noise_alpha=5.0,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2
)

# 6. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=full_dataset,
    args=sft_config,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/33955 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [9]:
# 7. Start Run & Monitor VRAM
# W&B automatically picks up GPU metrics, but if it lags,
# you can see it in the Colab "Resources" tab in real-time.
print("🚀 Launching Final Production Run...")
trainer.train()
wandb.finish()

🚀 Launching Final Production Run...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 33,955 | Num Epochs = 1 | Total steps = 531
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Step,Training Loss
1,3.647680
2,3.621097
3,3.364116
4,3.187878
5,3.201735
6,2.816010
7,2.601135
8,2.330219
9,1.957508
10,1.687374


Step,Training Loss
1,3.647680
2,3.621097
3,3.364116
4,3.187878
5,3.201735
6,2.816010
7,2.601135
8,2.330219
9,1.957508
10,1.687374


train/epoch,▁▁▁▁▂▂▂▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇██████
train/grad_norm,█▅▅▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▁▅██▇▇▇▇▇▆▆▅▅▅▅▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁
train/loss,█▇▅▅▄▃▃▂▃▄▃▂▂▂▃▂▂▃▂▂▃▂▂▂▂▂▂▂▂▁▁▂▁▃▁▂▂▂▁▂
total_flos,1.820548305895588e+17
train/epoch,1
train/global_step,531
train/grad_norm,0.41403
train/learning_rate,0.0
train/loss,0.70711


In [11]:
# 8. Merge and Compress to 4-bit (PTQ)
print("💾 Fusing and Quantizing for ZeroGPU Deployment...")
save_directory = "/content/drive/MyDrive/medical-llama3/production-4bit"
model.save_pretrained_merged(save_directory, tokenizer, save_method="merged_4bit_forced")

print("✅ COMPLETE. Your medical model is now ready for deployment.")

💾 Fusing and Quantizing for ZeroGPU Deployment...
Unsloth: Merging LoRA weights into 4bit model...


ValueError: Model does not appear to be quantized. Cannot use 'merged_4bit'.

In [12]:
# 8. The Rescue: Extracting the LoRA Adapters
print("💾 Saving LoRA adapters to Google Drive...")
save_directory = "/content/drive/MyDrive/medical-llama3/lora-production"

# This pulls ONLY your highly trained medical weights (approx. 50MB)
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print("✅ RESCUE COMPLETE. Your weights are permanently secured.")

💾 Saving LoRA adapters to Google Drive...
✅ RESCUE COMPLETE. Your weights are permanently secured.


# Compare original and fine-tuned LLM by removing and adding LORA adapters

In [14]:
# 1. The Open-Ended Inference Prompt
inference_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a highly accurate medical AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
""" # <--- Notice there is NO <|eot_id|> at the end of this string!

# 2. Define your test questions
test_prompts = [
    "What are the primary symptoms and first-line treatment for acute pericarditis?",
    "Explain the difference between Type 1 and Type 2 diabetes.",
    "What is the capital of France?"
]

# 3. The corrected generator
def generate_response(question):
    # Format using the open-ended inference prompt
    prompt = inference_prompt.format(question)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Generate the answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        use_cache=True,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode and clean up
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return response.split("assistant\n")[-1].strip()

# 4. Execute the A/B Test
print("🧪 STARTING CORRECTED LORA A/B TEST...\n")
print("-" * 50)

for q in test_prompts:
    print(f"🗣️ QUESTION: {q}\n")

    with model.disable_adapter():
        baseline_answer = generate_response(q)
        print(f"🤖 BASELINE LLAMA-3:\n{baseline_answer}\n")

    medical_answer = generate_response(q)
    print(f"🩺 YOUR MEDICAL MODEL:\n{medical_answer}\n")
    print("-" * 50)

🧪 STARTING CORRECTED LORA A/B TEST...

--------------------------------------------------
🗣️ QUESTION: What are the primary symptoms and first-line treatment for acute pericarditis?

🤖 BASELINE LLAMA-3:
Acute pericarditis is a sudden and severe inflammation of the pericardium, the sac-like membrane that surrounds the heart. The primary symptoms of acute pericarditis typically include:

1. Chest pain: The most common symptom of acute pericarditis is severe chest pain that is often sharp, stabbing, or aching in nature. The pain may be located in the center of the chest, and it may radiate to the neck, shoulder, or arm.
2. Shortness of breath: Patients with acute pericarditis may experience shortness of breath, which can be due to the inflammation and swelling of the pericardium, which can compress the heart and reduce its ability to fill with blood.
3

🩺 YOUR MEDICAL MODEL:
Acute pericarditis is a condition that involves inflammation of the pericardium, which is the sac-like membrane tha